# Binary Classification Data Preparation Pipeline (v2)

This notebook contains the data preparation steps to generate the datasets used for training the **Binary Classification Models** (DistilBERT and RoBERTa Baselines). It builds four sequential dataset iterations:

1. **First Dataset (`first_dataset.csv`)**: Baseline binary dataset combining self-reported mental health posts (Class 1) and general controls (Class 0), with explicit mental health keywords removed to avoid leakage.
2. **Second Dataset (`second_dataset.csv`)**: Length-augmented and class-balanced dataset adding further general controls.
3. **Third Dataset (`third_dataset.csv`)**: Curated binary dataset focusing on self-reported clinical posts vs. relevant topic controls.
4. **Fourth Dataset (`fourth_dataset.csv`)**: Augmented version of the third dataset incorporating synthetic samples.

---

## Setup & Global Imports

In [1]:
import pandas as pd
import re
import numpy as np

## Section 1: First Dataset Generation (`first_dataset.csv`)
Generate a baseline binary dataset by combining self-reported mental health posts and selected control posts, ensuring keyword filtering to mitigate target leakage.

### 1.1 Load Self-Reported Posts

In [2]:
user_df = pd.read_csv('self_reported_df.csv')
print("Self-Reported Shape:", user_df.shape)
print("Subreddit Counts:\n", user_df['subreddit'].value_counts())

Self-Reported Shape: (24015, 10)
Subreddit Counts:
 subreddit
adhd               4629
anxiety            3740
depression         3341
mentalhealth       3009
relationships      1694
socialanxiety      1179
bpd                1094
ptsd               1004
suicidewatch        704
autism              644
schizophrenia       520
healthanxiety       469
bipolarreddit       392
edanonymous         348
lonely              337
meditation          270
divorce             197
addiction           197
alcoholism          144
conspiracy           60
covid19_support      37
jokes                 6
Name: count, dtype: int64


### 1.2 Define Leakage Keywords to Remove
Define list of terms (subreddits, disorders, clinical terms) that could leak the target labels to the model.

In [3]:
words_to_remove = [
    "depression", "depressive", "depressed",
    "bipolar", "mania", "manic",
    "schizophrenia", "psychosis", "psychotic",
    "ocd", "obsessive compulsive",
    "ptsd", "post traumatic stress",
    "autism", "autistic",
    "adhd", "add",
    "bpd", "borderline",
    "anxiety", "anxious",
    "panic attack", "panic disorder",
    "social anxiety", "generalized anxiety",
    "eating disorder", "anorexia", "bulimia", "binge eating",
    "insomnia", "sleep disorder",
    "substance abuse", "substance use disorder",
    "suicide", "suicidal", "suicidewatch",
    "self harm", "cutting",
    "therapy", "therapist", "psychiatrist", "psychologist",
    "antidepressant", "ssri", "lexapro", "prozac", "zoloft",
    "adderall", "ritalin", "xanax", "valium", "lithium",
    "mental health professional", "diagnosed with", "diagnosis",
    "mental health", "mental health crisis"
]

# Build regular expression pattern for term removal
pattern = r"(" + "|".join(map(re.escape, words_to_remove)) + r")"

### 1.3 Clean Self-Reported Posts

In [4]:
# Clean self-reported text
user_df['other_posts'] = user_df['other_posts'].str.lower()
user_df['other_posts'] = user_df['other_posts'].str.replace(
    pattern,
    "",
    case=False,
    regex=True
)

### 1.4 Select and Clean SuicideWatch Posts from Control Subreddits

In [5]:
mental_subreddits_control = pd.read_csv('mental_disorders_subreddits_control.csv')
print("Subreddit Counts in Control Dataset:\n", mental_subreddits_control['subreddit'].value_counts())

# Select suicidewatch posts
suicide_watch = mental_subreddits_control.loc[mental_subreddits_control['subreddit'] == 'suicidewatch'].copy()

# Remove leakage keywords
suicide_watch['other_posts'] = suicide_watch['other_posts'].str.lower()
suicide_watch['other_posts'] = suicide_watch['other_posts'].str.replace(
    pattern,
    "",
    case=False,
    regex=True
)

C:\Users\hana\AppData\Local\Temp\ipykernel_45376\1499705465.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  mental_subreddits_control = pd.read_csv('mental_disorders_subreddits_control.csv')


Subreddit Counts in Control Dataset:
 subreddit
depression       26106
suicidewatch     14369
mentalhealth      9975
adhd              8897
bpd               7288
anxiety           6890
edanonymous       6238
socialanxiety     5304
alcoholism        2236
healthanxiety     2175
autism            2117
schizophrenia     1951
addiction         1874
ptsd              1051
bipolarreddit      887
Name: count, dtype: int64


### 1.5 Handle Missing (NaN) Values in Self-Reported and SuicideWatch Data
Optimised with vectorized `.fillna()` statements instead of nested `.iterrows()` loops.

In [6]:
# Vectorized missing values fill for user_df
user_df['report_post'] = user_df['report_post'].fillna(' ')
user_df['other_posts'] = user_df['other_posts'].fillna(' ')

# Vectorized missing values fill for suicide_watch
suicide_watch = suicide_watch.fillna(' ')
suicide_watch['report_post'] = ' '

### 1.6 Concatenate and Label At-Risk Group (Class 1)

In [7]:
new = pd.concat([user_df, suicide_watch], ignore_index=True)
new['label'] = 1

### 1.7 Load and Clean Irrelevant Topic Controls (Class 0)

In [8]:
non_mental_health = pd.read_csv('controls_irrelevent_topics.csv')

# Vectorized missing values fill for control topics
non_mental_health = non_mental_health.fillna(' ')
non_mental_health['label'] = 0

# Select top 40,000 controls
non_mental_health_sub = non_mental_health.iloc[:40000]

### 1.8 Build and Save First Dataset (`first_dataset.csv`)
Concatenate at-risk group and control group.

In [9]:
final = pd.concat([new, non_mental_health_sub], ignore_index=True)
final['other_posts'] = final['other_posts'].str.lower()

# Note: Cell 23 from the original notebook contained redundant code that modified
# user_df but had no effect on the concatenated 'final' dataframe. We skip it here to keep the code clean.

final.to_csv('first_dataset.csv', index=False)
print("First Dataset Created! Shape:", final.shape)

First Dataset Created! Shape: (78384, 11)


## Section 2: Second Dataset Generation (`second_dataset.csv`)
Augment the control class using a shuffled subset of relevant topics control.

In [10]:
first_dataset = pd.read_csv('first_dataset.csv')
relvent_topics = pd.read_csv('non_mental_disorders_relevent_subreddits_control.csv')

print("First Dataset Shape:", first_dataset.shape)
print("Relevant Topics Shape:", relvent_topics.shape)

# Shuffle and prepare controls
relvent_topics_control = relvent_topics.sample(random_state=42, frac=1)
relvent_topics_control['label'] = 0
relvent_topics_control_subset = relvent_topics_control.iloc[:20000].copy()

# Clean first dataset index
if 'Unnamed: 0' in first_dataset.columns:
    first_dataset = first_dataset.drop(columns=['Unnamed: 0'])

# Concatenate to form second dataset
second_dataset = pd.concat([first_dataset, relvent_topics_control_subset], ignore_index=True)
second_dataset = second_dataset.sample(random_state=42, frac=1).reset_index(drop=True)
second_dataset.to_csv('second_dataset.csv', index=False)
print("Second Dataset Created! Shape:", second_dataset.shape)

C:\Users\hana\AppData\Local\Temp\ipykernel_45376\3892855843.py:1: DtypeWarning: Columns (6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  first_dataset = pd.read_csv('first_dataset.csv')
C:\Users\hana\AppData\Local\Temp\ipykernel_45376\3892855843.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  relvent_topics = pd.read_csv('non_mental_disorders_relevent_subreddits_control.csv')


First Dataset Shape: (78384, 11)
Relevant Topics Shape: (100511, 10)
Second Dataset Created! Shape: (98384, 11)


## Section 3: Third Dataset Generation (`third_dataset.csv`)
Curate a dataset focusing strictly on self-reported clinical posts vs relevant topic controls.

In [11]:
suicidewatch = first_dataset[first_dataset['subreddit'] == 'suicidewatch']
irrelevant = first_dataset[first_dataset['label'] == 0]

# Filter for non-suicidewatch self-reported clinical posts
user_only_postprocessing = first_dataset[(first_dataset['label'] != 0) & (first_dataset['subreddit'] != 'suicidewatch')].copy()

# Note: In the original notebook, the following line was included but not assigned to a variable:
# pd.concat([user_only_postprocessing, suicidewatch[:1000]], ignore_index=True)
# To remain perfectly consistent with the existing third_dataset.csv, we preserve this behavior
# (meaning user_only_postprocessing remains without these 1,000 suicidewatch posts).

user_only_postprocessing = user_only_postprocessing.sample(frac=1, random_state=42).reset_index(drop=True)
user_only_postprocessing.to_csv('user_only_postprocessing.csv', index=False)

# Concatenate self-report, relevant controls, and a subset of general controls
third_dataset = pd.concat([
    user_only_postprocessing,
    relvent_topics_control.iloc[:23000],
    irrelevant.iloc[:1000]
], ignore_index=True)
third_dataset = third_dataset.sample(random_state=42, frac=1).reset_index(drop=True)

third_dataset.to_csv('third_dataset.csv', index=False)
print("Third Dataset Created! Shape:", third_dataset.shape)
print("Class Distribution:\n", third_dataset['label'].value_counts())

Third Dataset Created! Shape: (47311, 11)
Class Distribution:
 label
0    24000
1    23311
Name: count, dtype: int64


## Section 4: Fourth Dataset Generation (`fourth_dataset.csv`)
Augment the third dataset with synthetic samples (`syn_data.csv`).

In [12]:
syn_data = pd.read_csv('syn_data.csv')
third_dataset = pd.read_csv('third_dataset.csv')

print("Synthetic Data Shape:", syn_data.shape)
syn_data = syn_data.sample(random_state=42, frac=1)
syn_data = syn_data.drop_duplicates()

# Combine third dataset with a subset of synthetic samples
fourth_dataset = pd.concat([third_dataset, syn_data.iloc[:400]], ignore_index=True)
fourth_dataset = fourth_dataset.sample(random_state=42, frac=1).reset_index(drop=True)

fourth_dataset.to_csv('fourth_dataset.csv', index=False)
print("Fourth Dataset Created! Shape:", fourth_dataset.shape)
print("Class Distribution:\n", fourth_dataset['label'].value_counts())

Synthetic Data Shape: (473, 2)
Fourth Dataset Created! Shape: (47711, 11)
Class Distribution:
 label
0    24400
1    23311
Name: count, dtype: int64
